# RAG from Scratch to Production — Episode 16/20
## Evaluating RAG: one test suite for everything

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/JakkulaMahendhar/rag-from-scratch/blob/main/notebooks/16_evaluating_rag.ipynb)

Fifteen episodes, and almost every one ended the same way: *"measure it on your own documents"*. This episode builds the thing that makes that easy: **one test suite** you run after every change.

It also keeps a promise. On Episode 3, a reader suggested two checks:
> 1. Fail ingestion when the characters per page drop below a threshold (scans that report success).
> 2. Keep a few known facts per document, and check that retrieval can still find them after every parser change.

Both are in here.

**In this episode:**
1. Build a 4-layer suite: **ingestion → retrieval → answers → the quality badge**.
2. Run it on a known-good setup.
3. Make a realistic change (swap in the production parser), run it again, and see what the suite catches.

**Needs an LLM** for the answer layer: Gemini (free key) or local Ollama. About 30 calls.

## 0. Setup

In [ ]:
%pip install -q pymupdf python-docx google-generativeai requests sentence-transformers rank-bm25

In [1]:
import os, re, json, time, requests
from pathlib import Path
import numpy as np
import fitz                      # PyMuPDF
from docx import Document

PROVIDER = os.getenv("LLM_PROVIDER", "gemini")   # "gemini" or "ollama"
GEMINI_MODEL = "gemini-3.6-flash"
OLLAMA_MODEL = "llama3.1"


def _gemini_key():
    try:
        from google.colab import userdata  # Colab secrets
        return userdata.get("GEMINI_API_KEY")
    except Exception:
        return os.getenv("GEMINI_API_KEY") or __import__("getpass").getpass("GEMINI_API_KEY: ")


if PROVIDER == "gemini":
    import google.generativeai as genai
    genai.configure(api_key=_gemini_key())
    _model = genai.GenerativeModel(GEMINI_MODEL)

LLM_CALLS = {"count": 0, "seconds": 0.0}


def generate(prompt: str) -> str:
    start = time.perf_counter()
    if PROVIDER == "gemini":
        time.sleep(6)   # stay under the free tier's rate limit
        text = _model.generate_content(prompt, generation_config={"temperature": 0}).text
    else:
        r = requests.post("http://localhost:11434/api/generate",
                          json={"model": OLLAMA_MODEL, "prompt": prompt, "stream": False, "options": {"temperature": 0}},
                          timeout=300)
        r.raise_for_status()
        text = r.json()["response"]
    LLM_CALLS["count"] += 1
    LLM_CALLS["seconds"] += time.perf_counter() - start - (6 if PROVIDER == "gemini" else 0)
    return text.strip()


generate("Say OK")   # warm-up
LLM_CALLS.update(count=0, seconds=0.0)
print(f"Using {PROVIDER}")

Using ollama


## 1. The documents

Episode 3's four files, plus one new one: a scan with a **stamp** on it. A real scanner or mailroom often adds a line of text ("Scanned 12/03/2026"), so the page isn't completely empty, which is exactly what fools Episode 3's "empty page" check.

In [2]:
DOCS = Path("eval_docs")
DOCS.mkdir(exist_ok=True)

# 1. A normal 2-page PDF, with a header and footer on every page
pdf = fitz.open()
for i, body in enumerate([
    "Remote Work\nEmployees may work remotely up to 2 days per week. Wednesday is a mandatory in-office day for all "
    "teams. Remote days must be logged in the HR portal by Friday of the previous week.",
    "Annual Leave\nFull-time employees receive 24 days of paid annual leave per calendar year. A maximum of 5 unused "
    "days may be carried over into the next year.",
], start=1):
    page = pdf.new_page()
    page.insert_text((72, 40), "NovaTech Solutions - CONFIDENTIAL", fontsize=9)
    page.insert_textbox(fitz.Rect(72, 80, 520, 760), body, fontsize=12)
    page.insert_text((270, 810), f"Page {i} of 2", fontsize=9)
pdf.save(DOCS / "handbook.pdf")


def scanned_pdf(path, text, stamp=None):
    """Render text to an image (a 'scan'), optionally with a small text stamp on top."""
    original = fitz.open()
    page = original.new_page()
    page.insert_textbox(fitz.Rect(72, 72, 520, 400), text, fontsize=16)
    picture = page.get_pixmap(dpi=100)
    scan = fitz.open()
    scan_page = scan.new_page()
    scan_page.insert_image(fitz.Rect(0, 0, 595, 842), stream=picture.tobytes("png"))
    if stamp:
        scan_page.insert_text((400, 820), stamp, fontsize=8)
    scan.save(path, deflate=True)

# 2. A pure scan, and 3. a scan with a mailroom stamp
scanned_pdf(DOCS / "parental_leave.pdf", "Parental Leave\nNew parents receive 16 weeks of fully paid leave.")
scanned_pdf(DOCS / "it_notice.pdf", "IT Notice\nVPN error E-4031 means your access token expired. Reset it in the Okta portal.",
            stamp="Scanned 12/03/2026 - Mailroom")

# 4. A Word file where the amounts are in a table
doc = Document()
doc.add_heading("Employee Benefits", level=1)
doc.add_paragraph("NovaTech pays the following monthly benefits to every employee.")
table = doc.add_table(rows=1, cols=2)
table.rows[0].cells[0].text, table.rows[0].cells[1].text = "Benefit", "Amount"
for benefit, amount in [("Gym membership", "50 USD per month"), ("Phone bill", "30 USD per month"),
                        ("Internet at home", "25 USD per month")]:
    row = table.add_row().cells
    row[0].text, row[1].text = benefit, amount
doc.save(DOCS / "benefits.docx")

# 5. A text file saved by a Windows tool (not UTF-8)
(DOCS / "cafeteria.txt").write_bytes("Cafeteria\nLunch vouchers are worth 8 € per working day.\n".encode("cp1252"))

print(sorted(p.name for p in DOCS.iterdir()))

['benefits.docx', 'cafeteria.txt', 'handbook.pdf', 'it_notice.pdf', 'parental_leave.pdf']


## 2. Two parsers to compare

The **fixed** parser from Episode 3 (`parse_v2`), and the **production** parser (`app/services/parser.py`) as it is today. In the regression test, imagine someone "simplified" the code back to the production version: does the suite notice?

In [3]:
def parse_production(path: Path) -> str:
    """The production app's parser, unchanged."""
    if path.suffix == ".pdf":
        with fitz.open(path) as document:
            return "".join(page.get_text() for page in document)
    if path.suffix == ".docx":
        return "\n".join(p.text for p in Document(path).paragraphs)
    with open(path, "r", encoding="utf-8") as f:
        return f.read()


def parse_fixed(path: Path) -> str:
    """Episode 3's fixes: no headers/footers, tables kept, encoding fallback, scans fail loudly."""
    if path.suffix == ".pdf":
        with fitz.open(path) as document:
            pages = [p.get_text() for p in document]
            images = [bool(p.get_images()) for p in document]
        for n, (text, image) in enumerate(zip(pages, images), start=1):
            if not text.strip() and image:
                raise ValueError(f"page {n} is a scanned image with no text: needs OCR")
        lines = [{l.strip() for l in p.splitlines() if l.strip()} for p in pages]
        repeated = set.intersection(*lines) if len(pages) > 1 else set()
        kept = [l.strip() for p in pages for l in p.splitlines()
                if l.strip() and l.strip() not in repeated and not re.match(r"^page \d+( of \d+)?$", l.strip(), re.I)]
        merged = []
        for line in kept:
            if merged and len(merged[-1]) > 40 and not merged[-1].endswith((".", "!", "?", ":")):
                merged[-1] += " " + line
            else:
                merged.append(line)
        return "\n".join(merged)
    if path.suffix == ".docx":
        blocks = []
        for block in Document(path).iter_inner_content():
            if hasattr(block, "rows"):
                for row in block.rows[1:]:
                    cells = [c.text.strip() for c in row.cells]
                    blocks.append(f"{cells[0]}: {', '.join(cells[1:])}.")
            elif block.text.strip():
                blocks.append(block.text.strip())
        return "\n".join(blocks)
    raw = path.read_bytes()
    for encoding in ("utf-8-sig", "cp1252"):
        try:
            return raw.decode(encoding)
        except UnicodeDecodeError:
            pass
    return raw.decode("utf-8", errors="replace")

## 3. The test suite

### Layer 1: ingestion checks

Run on every uploaded file, before anything is indexed:
- **Did parsing fail?** Then the file isn't searchable at all.
- **Characters per page** (the reader's idea): a normal text page has hundreds of characters, a scan has almost none. Below **100**, flag it, even if parsing "succeeded".

In [4]:
MIN_CHARS_PER_PAGE = 100

def ingestion_check(parser):
    texts, problems = {}, []
    for path in sorted(DOCS.iterdir()):
        try:
            texts[path.name] = parser(path)
        except Exception as e:
            problems.append(f"{path.name}: parsing failed ({type(e).__name__})")
            continue
        if path.suffix == ".pdf":
            with fitz.open(path) as document:
                for n, page in enumerate(document, start=1):
                    chars = len(page.get_text().strip())
                    if chars < MIN_CHARS_PER_PAGE:
                        problems.append(f"{path.name}: page {n} has only {chars} characters (scan?)")
        elif not texts[path.name].strip():
            problems.append(f"{path.name}: no text extracted")
    return texts, problems

### Layer 2: known facts

For every document, a few questions whose answers we know are in it (the reader's second idea). After chunking and indexing, **search must find a chunk containing the fact**. No LLM needed: fast, free, and it runs after every parser, chunker or embedding change.

In [5]:
KNOWN_FACTS = [
    # (document, question, text the retrieved chunks must contain, words a correct answer contains)
    ("handbook.pdf",       "How many days per week can I work remotely?",            "2 days",   ["2"]),
    ("handbook.pdf",       "When must remote days be logged?",                       "friday",   ["friday"]),
    ("handbook.pdf",       "How many days of annual leave do full-time staff get?",  "24 days",  ["24"]),
    ("handbook.pdf",       "How many unused leave days can be carried over?",        "5 unused", ["5"]),
    ("benefits.docx",      "How much is the gym membership benefit?",                "50 usd",   ["50"]),
    ("benefits.docx",      "How much is paid towards the phone bill?",               "30 usd",   ["30"]),
    ("benefits.docx",      "How much is the internet at home benefit?",              "25 usd",   ["25"]),
    ("cafeteria.txt",      "How much are lunch vouchers worth?",                     "8 €",      ["8"]),
    ("parental_leave.pdf", "How many weeks of parental leave do new parents get?",   "16 weeks", ["16"]),
    ("it_notice.pdf",      "How do I fix VPN error E-4031?",                         "okta",     ["okta"]),
]
NOT_IN_DOCUMENTS = [
    "How many sick days do I get per year?", "Is there a company pension plan?",
    "What is the capital of France?", "What is the dress code for the office party?",
]

from sentence_transformers import SentenceTransformer
embedder = SentenceTransformer("all-MiniLM-L6-v2")


def build_index(texts):
    """Chunk every document into groups of 2 sentences, and embed them."""
    chunks = []
    for name, text in texts.items():
        sentences = [s.strip() for s in re.split(r"(?<=[.!?])\s+|\n+", text) if s.strip()]
        chunks += [" ".join(sentences[i:i + 2]) for i in range(0, len(sentences), 2)]
    return chunks, embedder.encode(chunks, normalize_embeddings=True) if chunks else None


def search(question, index, k=3):
    chunks, vectors = index
    if not chunks:
        return []
    return [chunks[i] for i in np.argsort(-(vectors @ embedder.encode(question, normalize_embeddings=True)))[:k]]


def known_facts_check(index):
    found, missing = 0, []
    for doc, question, fact, _ in KNOWN_FACTS:
        if any(fact in c.lower() for c in search(question, index)):
            found += 1
        else:
            missing.append(f"{doc}: {question}")
    return found, missing

### Layer 3: answers

End to end through the LLM, with production's prompt rules: the 10 known-fact questions must be **answered**, and 4 questions that aren't in any document must be **refused**.

In [6]:
FALLBACK = "I don't have enough information from the provided documents."

def answer(question, index):
    context = "\n\n".join(search(question, index)) or "(no relevant context found)"
    return generate(f"""You are an AI assistant specialized in answering questions from provided documents.
Follow these rules strictly:
1. Answer only using the provided context.
2. Do not use external knowledge.
3. If the answer is not available in the context, respond exactly:
"{FALLBACK}"
4. Do not create or assume facts.
5. Keep the answer clear and concise.

Context:
{context}

Question: {question}
Answer:""")


def answer_check(index):
    answered = sum(
        (lambda a: "enough information" not in a.lower() and any(w in a.lower() for w in words))(answer(q, index))
        for _, q, _, words in KNOWN_FACTS)
    refused = sum("enough information" in answer(q, index).lower() for q in NOT_IN_DOCUMENTS)
    return answered, refused

### The whole suite

In [7]:
def run_suite(name, parser):
    LLM_CALLS.update(count=0, seconds=0.0)
    texts, problems = ingestion_check(parser)
    index = build_index(texts)
    facts_found, facts_missing = known_facts_check(index)
    answered, refused = answer_check(index)

    print(f"══ {name}")
    print(f"Layer 1 · ingestion problems: {len(problems)}")
    for p in problems:
        print(f"      ⚠️ {p}")
    print(f"Layer 2 · known facts retrievable: {facts_found}/{len(KNOWN_FACTS)}")
    for m in facts_missing:
        print(f"      ❌ {m}")
    print(f"Layer 3 · answered correctly: {answered}/{len(KNOWN_FACTS)}   refused correctly: {refused}/{len(NOT_IN_DOCUMENTS)}")
    print(f"Cost · {LLM_CALLS['count']} LLM calls, {LLM_CALLS['seconds']:.1f}s\n")
    return {"problems": len(problems), "facts": facts_found, "answered": answered, "refused": refused}

## 4. Run 1: the known-good setup

In [8]:
BASELINE = run_suite("Fixed parser (baseline)", parse_fixed)

══ Fixed parser (baseline)
Layer 1 · ingestion problems: 2
      ⚠️ it_notice.pdf: page 1 has only 29 characters (scan?)
      ⚠️ parental_leave.pdf: parsing failed (ValueError)
Layer 2 · known facts retrievable: 8/10
      ❌ parental_leave.pdf: How many weeks of parental leave do new parents get?
      ❌ it_notice.pdf: How do I fix VPN error E-4031?
Layer 3 · answered correctly: 8/10   refused correctly: 4/4
Cost · 14 LLM calls, 5.9s


Nothing is perfect, and the suite says so precisely:
- **Both scans are flagged.** The pure scan fails loudly (Episode 3's fix). The **stamped** scan parses "successfully", but has only a few dozen characters on its page: the characters-per-page check catches what the empty-page check missed.
- The facts in those two scans aren't retrievable. That's correct: they need OCR, and now someone knows it.
- Everything else is found and answered.

This is the **baseline**: the numbers to compare every future change against.

## 5. Run 2: a "harmless" change

Someone replaces the parser with a simpler version: the production one. The app still starts, and every upload returns "completed" or a single error. Would anyone notice?

In [9]:
REGRESSION = run_suite("Production parser (the change)", parse_production)

══ Production parser (the change)
Layer 1 · ingestion problems: 3
      ⚠️ cafeteria.txt: parsing failed (UnicodeDecodeError)
      ⚠️ it_notice.pdf: page 1 has only 29 characters (scan?)
      ⚠️ parental_leave.pdf: page 1 has only 0 characters (scan?)
Layer 2 · known facts retrievable: 4/10
      ❌ benefits.docx: How much is the gym membership benefit?
      ❌ benefits.docx: How much is paid towards the phone bill?
      ❌ benefits.docx: How much is the internet at home benefit?
      ❌ cafeteria.txt: How much are lunch vouchers worth?
      ❌ parental_leave.pdf: How many weeks of parental leave do new parents get?
      ❌ it_notice.pdf: How do I fix VPN error E-4031?
Layer 3 · answered correctly: 4/10   refused correctly: 4/4
Cost · 14 LLM calls, 5.6s


In [10]:
print(f"{'':<30} {'baseline':>9} {'after change':>13}")
for key, label, total in [("problems", "ingestion problems flagged", None),
                          ("facts", "known facts retrievable", len(KNOWN_FACTS)),
                          ("answered", "answered correctly", len(KNOWN_FACTS)),
                          ("refused", "refused correctly", len(NOT_IN_DOCUMENTS))]:
    suffix = f"/{total}" if total else ""
    flag = "  ⬅ regression" if total and REGRESSION[key] < BASELINE[key] else ""
    print(f"{label:<30} {str(BASELINE[key]) + suffix:>9} {str(REGRESSION[key]) + suffix:>13}{flag}")

                                baseline  after change
ingestion problems flagged             2             3
known facts retrievable             8/10          4/10  ⬅ regression
answered correctly                  8/10          4/10  ⬅ regression
refused correctly                    4/4           4/4


The suite catches the regression in seconds:
- **Layer 1** flags the text file that now crashes on `€`.
- **Layer 2** pinpoints which facts disappeared: the three benefit amounts (the Word table is gone) and the lunch vouchers. No LLM needed to see that.
- **Layer 3** confirms what users would feel: those questions now get "I don't have enough information".

Without the suite, this change ships, and the first sign is a user asking about their gym benefit.

## 6. Layer 4: can you trust the quality badge?

The [production app](https://github.com/JakkulaMahendhar/rag-chatbot) shows users an answer-quality badge (**Excellent / Good / Weak**) next to each answer, from `app/services/search_evaluator.py`: it grades the **best hybrid search score** (Episode 8's formula). Does the badge agree with whether the answer is actually right?

In [11]:
from rank_bm25 import BM25Okapi

texts, _ = ingestion_check(parse_fixed)
index = build_index(texts)
chunks, vectors = index
bm25 = BM25Okapi([c.lower().split() for c in chunks])          # production tokenizer (Episode 7)

def production_badge(question):
    cosine = vectors @ embedder.encode(question, normalize_embeddings=True)
    vector = 1 / (1 + (2 - 2 * cosine))                          # production's distance → score
    keyword = np.array(bm25.get_scores(question.lower().split()))
    keyword = keyword / keyword.max() if keyword.max() > 0 else keyword
    best = float((0.6 * vector + 0.4 * keyword).max())
    return ("Excellent" if best >= 0.7 else "Good" if best >= 0.5 else "Weak"), best

print(f"{'badge':<10} {'score':>5}  {'is the answer right?':<22} question")
for doc, q, _, words in KNOWN_FACTS:
    reply = answer(q, index)
    right = "enough information" not in reply.lower() and any(w in reply.lower() for w in words)
    badge, best = production_badge(q)
    print(f"{badge:<10} {best:>5.2f}  {'✅ answered' if right else '❌ not answered':<22} {q}")
for q in NOT_IN_DOCUMENTS:
    badge, best = production_badge(q)
    print(f"{badge:<10} {best:>5.2f}  {'(not in the documents)':<22} {q}")

badge      score  is the answer right?   question


Excellent   0.86  ✅ answered             How many days per week can I work remotely?


Excellent   0.78  ✅ answered             When must remote days be logged?


Excellent   0.82  ✅ answered             How many days of annual leave do full-time staff get?


Excellent   0.81  ✅ answered             How many unused leave days can be carried over?


Excellent   0.77  ✅ answered             How much is the gym membership benefit?


Good        0.67  ✅ answered             How much is paid towards the phone bill?


Excellent   0.79  ✅ answered             How much is the internet at home benefit?


Excellent   0.81  ✅ answered             How much are lunch vouchers worth?


Good        0.69  ❌ not answered         How many weeks of parental leave do new parents get?


Weak        0.22  ❌ not answered         How do I fix VPN error E-4031?
Good        0.65  (not in the documents) How many sick days do I get per year?
Good        0.62  (not in the documents) Is there a company pension plan?
Good        0.59  (not in the documents) What is the capital of France?
Good        0.63  (not in the documents) What is the dress code for the office party?


The badge measures **how well the search matched words**, not whether the answer is right:
- Questions whose answer is missing (the scans) can still get a **"Good"** badge.
- Questions that aren't in the documents at all, like *"What is the capital of France?"*, get **"Good"** too (Episode 8: the top keyword match always scores 1.0).

A badge users rely on should be tested like any other feature, against known answers. The production app also has a second evaluator, `rag_evaluator.py`, that returns **"Good" for every answer that has a source**, whatever the answer says.

## 7. What production does

The [production app](https://github.com/JakkulaMahendhar/rag-chatbot) has no evaluation suite. Its two "evaluators" grade search scores, not answers. Everything in this episode can run in CI (Episode 20) on every pull request: layers 1 and 2 need no LLM at all.

## Recap: what to measure, and how

| Layer | What it catches | Needs an LLM? | Series episode |
|---|---|---|---|
| Ingestion: parse errors, chars per page | Crashes, scans that report success | No | 3 |
| Known facts retrievable | Lost tables, bad chunking, tokenizer bugs | No | 4, 7 |
| Answers and refusals | What users actually see | Yes | 1, 11, 13 |
| Quality signals vs reality | Badges and scores that don't mean what they say | Yes | 8, 11 |

**The rule:** write down what "working" means for **your** documents (facts that must be found, questions that must be refused), and run it after every change. Thanks to the reader on Episode 3 for both ideas.

**Next, Episode 17: FastAPI, streaming and the background worker**, the parts that turn this pipeline into an app.

⭐ Notebooks: https://github.com/JakkulaMahendhar/rag-from-scratch
🏗️ Full production app: https://github.com/JakkulaMahendhar/rag-chatbot
📸 Instagram: [@codingwithmahi](https://instagram.com/codingwithmahi)